In [1]:
import pandas as pd 
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import statsmodels.api as sm
import pickle

In [2]:
df = pd.read_csv("gurgaon_properties_post_feature_selection_v2.csv").drop(
    columns=['store room', 'floor_category', 'balcony'], errors='ignore'
)

In [3]:
age_map = {
    'Relatively New':     'new',
    'New Property':       'new',
    'Moderately old':     'old',
    'Old Property':       'old',
    'Under Construction': 'under construction',
}
df['agePossession'] = df['agePossession'].map(age_map).fillna(df['agePossession'])

In [4]:
df['property_type']   = df['property_type'].map({'flat': 0, 'house': 1})
df['luxury_category'] = df['luxury_category'].map({'Low': 0, 'Medium': 1, 'High': 2})

In [5]:
new_df = pd.get_dummies(df, columns=['sector', 'agePossession'], drop_first=True)

X = new_df.drop(columns='price')
y = new_df['price']
y_log = np.log1p(y)

In [6]:
new_df.head()

,property_type,price,bedRoom,bathroom,built_up_area,servent room,furnishing_type,luxury_category,sector_sector 10,sector_sector 102,...,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,agePossession_old,agePossession_under construction
0,0,0.21,1.0,1,336.0,0,0,1,False,False,...,False,False,False,False,True,False,False,False,False,False
1,0,5.90,4.0,4,5350.0,0,0,1,False,False,...,False,False,False,False,False,False,False,False,True,False
2,0,0.90,3.0,3,1900.0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,True,False
3,1,10.00,5.0,5,4518.0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,True,False
4,0,0.72,2.0,2,1165.0,0,0,2,False,False,...,False,False,False,False,False,False,False,False,False,False


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)

X_test_scaled = scaler.transform(X_test)   # only transform, never fit
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [9]:
X_with_const = sm.add_constant(X_train_scaled)
ols_model = sm.OLS(y_train.values, X_with_const).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.850
Model:                            OLS   Adj. R-squared:                  0.845
Method:                 Least Squares   F-statistic:                     156.5
Date:                Sun, 22 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:40:27   Log-Likelihood:                 300.56
No. Observations:                2911   AIC:                            -395.1
Df Residuals:                    2808   BIC:                             220.4
Df Model:                         102                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [13]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV

# Find best alpha for Ridge
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 200]}
ridge_cv = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)

best_alpha = ridge_cv.best_params_['alpha']
print(f"Best Ridge alpha: {best_alpha}")

ridge_model = Ridge(alpha=best_alpha)
ridge_model.fit(X_train_scaled, y_train)

print(f"Ridge R² on test : {ridge_model.score(X_test_scaled, y_test):.4f}")
r2=ridge_model.score(X_test_scaled, y_test)
n=X_test.shape[0]
p=X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adujusted_r2",adj_r2)


Best Ridge alpha: 0.1
Ridge R² on test : 0.8231
Adujusted_r2 0.7942535373527284


In [21]:
coef_df=pd.DataFrame(ridge_model.coef_.reshape(1,102),columns=X_train_scaled.columns).stack().reset_index().drop(columns='level_0').rename(columns={'level_1':'features',0:'coef'})

In [32]:
pd.set_option('display.max_rows', None)
coef_df

,features,coef
0,property_type,0.066935
1,bedRoom,0.055282
2,bathroom,0.091545
3,built_up_area,0.211251
4,servent room,0.081257
5,furnishing_type,0.012306
6,luxury_category,0.011095
7,sector_sector 10,0.013466
8,sector_sector 102,0.069695
9,sector_sector 103,0.022594


In [33]:
insights_data = {}
median_price = y.median()

for feat in ['bedRoom', 'bathroom', 'built_up_area', 'servent room',
             'property_type', 'furnishing_type', 'luxury_category']:
    if feat not in X.columns:
        continue

    feat_idx     = list(X.columns).index(feat)
    original_std = scaler.scale_[feat_idx]

    beta = ols_model.params[feat_idx + 1]
    pval = ols_model.pvalues[feat_idx + 1]
    ci   = ols_model.conf_int().iloc[feat_idx + 1]

    log_effect = beta / original_std
    pct_change = np.expm1(log_effect) * 100

    insights_data[feat] = {
        'pct_change':      round(pct_change, 3),
        'abs_change_lakh': round(median_price * (pct_change / 100) * 100, 2),
        'p_value':         round(pval, 4),
        'significant':     bool(pval < 0.05),
        'ci_low_pct':      round(np.expm1(ci.iloc[0] / original_std) * 100, 3),
        'ci_high_pct':     round(np.expm1(ci.iloc[1] / original_std) * 100, 3),
        'direction':       'increases' if pct_change > 0 else 'decreases',
    }

# Verify before saving
for k, v in insights_data.items():
    print(f"{k:20s}: {v['pct_change']:+.2f}%  |  ₹{v['abs_change_lakh']:.1f}L  |  sig={v['significant']}")

bedRoom             : +4.49%  |  ₹6.7L  |  sig=True
bathroom            : +6.50%  |  ₹9.8L  |  sig=True
built_up_area       : +0.02%  |  ₹0.0L  |  sig=True
servent room        : +32.84%  |  ₹49.2L  |  sig=True
property_type       : +17.25%  |  ₹25.9L  |  sig=True
furnishing_type     : +1.40%  |  ₹2.1L  |  sig=True
luxury_category     : +1.60%  |  ₹2.4L  |  sig=True


C:\Users\sarth\AppData\Local\Temp\ipykernel_16768\2269154654.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta = ols_model.params[feat_idx + 1]
C:\Users\sarth\AppData\Local\Temp\ipykernel_16768\2269154654.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pval = ols_model.pvalues[feat_idx + 1]


In [35]:
insights_data = {}
median_price = y.median()

# Define meaningful unit per feature
# built_up_area: show per 100 sqft, not per 1 sqft
feature_units = {
    'bedRoom':          {'unit': 1,   'unit_label': 'bedroom'},
    'bathroom':         {'unit': 1,   'unit_label': 'bathroom'},
    'built_up_area':    {'unit': 100, 'unit_label': '100 sqft'},
    'servent room':     {'unit': 1,   'unit_label': 'servant room added'},
    'property_type':    {'unit': 1,   'unit_label': 'flat → house'},
    'furnishing_type':  {'unit': 1,   'unit_label': 'furnishing level'},
    'luxury_category':  {'unit': 1,   'unit_label': 'luxury level'},
}

for feat, meta in feature_units.items():
    if feat not in X.columns:
        continue

    feat_idx     = list(X.columns).index(feat)
    original_std = scaler.scale_[feat_idx]
    beta         = ols_model.params[feat_idx + 1]
    pval         = ols_model.pvalues[feat_idx + 1]
    ci           = ols_model.conf_int().iloc[feat_idx + 1]

    # Multiply log_effect by unit to scale interpretation correctly
    unit       = meta['unit']
    log_effect = (beta / original_std) * unit
    pct_change = np.expm1(log_effect) * 100

    abs_change_lakh = median_price * (pct_change / 100) * 100

    insights_data[feat] = {
        'pct_change':      round(pct_change, 3),
        'abs_change_lakh': round(abs_change_lakh, 2),
        'p_value':         round(pval, 4),
        'significant':     bool(pval < 0.05),
        'ci_low_pct':      round(np.expm1((ci.iloc[0] / original_std) * unit) * 100, 3),
        'ci_high_pct':     round(np.expm1((ci.iloc[1] / original_std) * unit) * 100, 3),
        'direction':       'increases' if pct_change > 0 else 'decreases',
        'unit_label':      meta['unit_label'],   # for Streamlit display
    }

# Print with unit context
print(f"\n{'Feature':<20} {'Unit':<22} {'% Change':>10} {'₹ Lakh':>8} {'Sig':>6}")
print("-" * 72)
for k, v in insights_data.items():
    print(f"{k:<20} {v['unit_label']:<22} {v['pct_change']:>+10.2f} "
          f"{v['abs_change_lakh']:>8.2f} {str(v['significant']):>6}")


# ---

# Now the output will read like this:
# ```
# Feature              Unit                   % Change    ₹ Lakh   Sig
# ------------------------------------------------------------------------
# bedRoom              bedroom                   +4.49%      6.7   True
# bathroom             bathroom                  +6.50%      9.8   True
# built_up_area        100 sqft                  +2.0%       3.0   True  ← fixed
# servent room         servant room added       +32.84%     49.2   True
# property_type        flat → house             +17.25%     25.9   True
# furnishing_type      furnishing level          +1.40%      2.1   True
# luxury_category      luxury level              +1.60%      2.4   True


Feature              Unit                     % Change   ₹ Lakh    Sig
------------------------------------------------------------------------
bedRoom              bedroom                     +4.49     6.73   True
bathroom             bathroom                    +6.50     9.76   True
built_up_area        100 sqft                    +1.68     2.52   True
servent room         servant room added         +32.84    49.25   True
property_type        flat → house               +17.25    25.88   True
furnishing_type      furnishing level            +1.40     2.10   True
luxury_category      luxury level                +1.60     2.40   True


C:\Users\sarth\AppData\Local\Temp\ipykernel_16768\414041194.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta         = ols_model.params[feat_idx + 1]
C:\Users\sarth\AppData\Local\Temp\ipykernel_16768\414041194.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pval         = ols_model.pvalues[feat_idx + 1]


In [38]:
pd.DataFrame(insights_data,columns=insights_data.keys())

,bedRoom,bathroom,built_up_area,servent room,property_type,furnishing_type,luxury_category
pct_change,4.49,6.504,1.681,32.835,17.251,1.399,1.603
abs_change_lakh,6.73,9.76,2.52,49.25,25.88,2.1,2.4
p_value,0.0,0.0,0.0,0.0,0.0,0.0097,0.02
significant,True,True,True,True,True,True,True
ci_low_pct,3.263,5.422,1.588,27.472,13.465,0.337,0.251
ci_high_pct,5.731,7.598,1.773,38.424,21.164,2.473,2.973
direction,increases,increases,increases,increases,increases,increases,increases
unit_label,bedroom,bathroom,100 sqft,servant room added,flat → house,furnishing level,luxury level


In [39]:
import pickle
with open('insights.pkl','wb') as file:
    pickle.dump(insights_data,file)
    

In [40]:
with open('insights.pkl','rb') as file:
    insights_test=pickle.load(file)

In [42]:
insights_data['bedRoom']

{'pct_change': np.float64(4.49),
 'abs_change_lakh': np.float64(6.73),
 'p_value': np.float64(0.0),
 'significant': True,
 'ci_low_pct': np.float64(3.263),
 'ci_high_pct': np.float64(5.731),
 'direction': 'increases',
 'unit_label': 'bedroom'}